# Conditional-NF Novel View Generator -- Colab training

Trains a **conditional Normalizing Flow** that models `P((3D point, 3D direction) | DINO feature)`,
against a **frozen** nerfacto NeRF (trained once here, then never fine-tuned again).

This repo is a public fork (`itayhanoch/VF-NeRF-conditional`), so no GitHub token is needed to clone it.

Steps: install deps (including compiling tiny-cuda-nn from source) -> mount Drive -> download/prepare
a scene -> train the frozen NeRF backbone -> precompute DINO features -> train the conditional NF,
checkpointing to Drive so it survives disconnects. `Runtime` -> `Run all` once `TRAIN_RESUME_MODE` (below)
is set.


In [ ]:
#@title Install dependencies
import os

REPO_URL = "https://github.com/itayhanoch/VF-NeRF-conditional.git"

if not os.path.isdir("VF-NeRF-conditional"):
    !git clone {REPO_URL}
%cd VF-NeRF-conditional

# This repo pins torch<2.0 (tiny-cuda-nn/nerfacc compatibility) and needs tiny-cuda-nn
# compiled from source -- matches the original VF-NeRF README's own instructions.
!pip install --quiet torch==1.13.1 torchvision functorch --extra-index-url https://download.pytorch.org/whl/cu117
!pip install --quiet ninja
!pip install --quiet "git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch"

!pip install --quiet --upgrade pip setuptools
!pip install --quiet -e .
!pip install --quiet -e ./normalizing-flows


In [ ]:
#@title Mount Google Drive (checkpoints persist here across disconnects)
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/vf_nerf_conditional"
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)


## Configuration

Set `TRAIN_RESUME_MODE` once, then `Run all` -- nothing later needs input.
Default target scene is one of nerfstudio's bundled example captures
(dataset-agnostic: point `SCENE_NAME`/`DATA_DIR` at your own COLMAP-processed
scene later with no code changes).


In [ ]:
#@title Configuration
SCENE_NAME = "poster"  #@param {type:"string"}
DATA_DIR = f"data/nerfstudio/{SCENE_NAME}"
NERF_OUTPUT_DIR = f"{DRIVE_ROOT}/nerf_outputs"
COND_NF_CHECKPOINT_DIR = f"{DRIVE_ROOT}/conditional_nf/{SCENE_NAME}"

# "finetune": resume+continue training if a checkpoint already exists on Drive
# "reset": always retrain from scratch
# "skip": load existing weights only, skip training entirely
TRAIN_RESUME_MODE = "finetune"  #@param ["finetune", "reset", "skip"]


In [ ]:
#@title Download the example scene (skip if DATA_DIR already exists, e.g. your own scene)
import os

if not os.path.isdir(DATA_DIR):
    !ns-download-data nerfstudio --capture-name={SCENE_NAME}
else:
    print(f"{DATA_DIR} already exists, skipping download.")


## Train the frozen NeRF backbone

Trained once, then frozen for everything downstream -- the conditional-NF
training script never fine-tunes it.


In [ ]:
#@title Train (or reuse) the frozen nerfacto backbone
import glob

existing = sorted(glob.glob(f"{NERF_OUTPUT_DIR}/{SCENE_NAME}/nerfacto/*/config.yml"))

if existing and TRAIN_RESUME_MODE == "skip":
    NERF_CONFIG = existing[-1]
    print(f"Using existing frozen-NeRF checkpoint: {NERF_CONFIG}")
else:
    !ns-train nerfacto \
        --data {DATA_DIR} \
        --output-dir {NERF_OUTPUT_DIR} \
        --viewer.quit-on-train-completion True \
        --vis tensorboard
    existing = sorted(glob.glob(f"{NERF_OUTPUT_DIR}/{SCENE_NAME}/nerfacto/*/config.yml"))
    NERF_CONFIG = existing[-1]
    print(f"Trained frozen-NeRF checkpoint: {NERF_CONFIG}")


## Train the conditional Normalizing Flow

Standalone script (`scripts/train_conditional_nf.py`) -- bypasses nerfstudio's
Trainer/Pipeline entirely: loads the frozen NeRF above purely for inference,
precomputes/caches DINOv2 features for every training image, then runs a plain
PyTorch training loop, checkpointing to Drive on an interval.


In [ ]:
#@title Train the conditional NF
if TRAIN_RESUME_MODE == "skip":
    print("TRAIN_RESUME_MODE == 'skip': not training, expecting an existing checkpoint at "
          f"{COND_NF_CHECKPOINT_DIR}/latest.pt")
else:
    if TRAIN_RESUME_MODE == "reset":
        import shutil
        shutil.rmtree(COND_NF_CHECKPOINT_DIR, ignore_errors=True)

    !python scripts/train_conditional_nf.py \
        --nerf-config {NERF_CONFIG} \
        --scene-dir {DATA_DIR} \
        --checkpoint-dir {COND_NF_CHECKPOINT_DIR} \
        --max-steps 20000 \
        --batch-size 4096


## Quick sanity render

Sample one condition from a random training-image pixel and render it through
the frozen NeRF, as a cheap end-to-end smoke test before running the full local
Gradio UI (`app/gradio_app.py`) against the downloaded checkpoints.


In [ ]:
#@title Quick sanity check
import random
from pathlib import Path

import torch

from nerfstudio.data.dataparsers.nerfstudio_dataparser import NerfstudioDataParserConfig
from nerfstudio.fields.nf_field import ConditionalNFField
from nerfstudio.utils.dino_features import DinoExtractor, get_or_compute_cache, load_image_chw_01
from nerfstudio.utils.eval_utils import eval_setup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_, pipeline, _, _ = eval_setup(Path(NERF_CONFIG), test_mode="inference")
nerf_model = pipeline.model.to(device).eval()

dataparser = NerfstudioDataParserConfig(data=Path(DATA_DIR)).setup()
outputs = dataparser.get_dataparser_outputs(split="train")
cameras = outputs.cameras.to(device)

ckpt = torch.load(f"{COND_NF_CHECKPOINT_DIR}/latest.pt", map_location=device)
field = ConditionalNFField(
    context_dim=ckpt["context_dim"], num_dims=ckpt["num_dims"], num_blocks=ckpt["num_blocks"],
    hidden_dim=ckpt["hidden_dim"], cond_prior=ckpt["cond_prior"], use_cond_in_coupling=True,
    use_batchnorm=ckpt["use_batchnorm"], device=str(device),
)
field.load_state_dict(ckpt["model_state"])
field.eval()

extractor = DinoExtractor(model_name=ckpt["dino_model_name"], device=str(device))
image_path = random.choice(outputs.image_filenames)
img = load_image_chw_01(image_path)
feat_map = get_or_compute_cache(img, image_path.stem, Path(DATA_DIR) / "dino_cache", extractor)
_, h, w = img.shape
y, x = h // 2, w // 2
condition = feat_map[:, y, x]

with torch.no_grad():
    samples = field.sample(num_samples=100, context=condition)
    log_p = field.log_prob(samples, condition.unsqueeze(0).expand(100, -1)).squeeze(-1)
best = samples[log_p.argmax()]
print(f"Best sample (position, direction): {best.tolist()}")
print(f"log-likelihood: {log_p.max().item():.3f}")
